# Endpoint Validation — Credit Card Fraud Detection

Tests the deployed real-time managed endpoint (`credit-fraud-detection-mbydj`) against a known legitimate transaction and a known fraudulent transaction, as part of UAT sign-off. See `docs/05_UAT_Checklist.md` for the full test matrix.

**Note:** the endpoint key below is redacted. Replace `<YOUR_ENDPOINT_KEY>` with the primary/secondary key from your own deployment's Consume tab before running.

In [ ]:
import urllib.request
import json
import os
import ssl

def allowSelfSignedHttps(allowed):
    # bypass the server certificate verification on client side
    if allowed and not os.environ.get('PYTHONHTTPSVERIFY', '') and getattr(ssl, '_create_unverified_context', None):
        ssl._create_default_https_context = ssl._create_unverified_context

allowSelfSignedHttps(True)  # this line is needed if you use self-signed certificate in your scoring service.

## Test 1 — Known Legitimate Transaction

Expected model output: `0` (not fraud).

In [ ]:
# Request data goes here
# The example below assumes JSON formatting which may be updated
# depending on the format your endpoint expects.
# More information can be found here:
# https://docs.microsoft.com/azure/machine-learning/how-to-deploy-advanced-entry-script
data = {
  "input_data": {
    "columns": [
      "Time", "V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8", "V9", "V10",
      "V11", "V12", "V13", "V14", "V15", "V16", "V17", "V18", "V19", "V20",
      "V21", "V22", "V23", "V24", "V25", "V26", "V27", "V28", "Amount"
    ],
    "index": [0],
    "data": [[
      0, -1.359807134, -0.072781173, 2.53634673, 1.378155224, -0.33832077,
      0.462387778, 0.239598554, 0.098697901, 0.36378697, 0.090794172,
      -0.551599533, -0.617800856, -0.991389847, -0.311169354, 1.468176972,
      -0.470400525, 0.207971242, 0.02579058, 0.40399296, 0.251412098,
      -0.018306778, 0.277837576, -0.11047391, 0.066928075, 0.128539358,
      -0.189114844, 0.133558377, -0.021053053, 149.62
    ]]
  },
  "params": {}
}

body = str.encode(json.dumps(data))

url = 'https://credit-fraud-detection-mbydj.<region>.inference.ml.azure.com/score'
# Replace this with the primary/secondary key, AMLToken, or Microsoft Entra ID token for the endpoint
api_key = '<YOUR_ENDPOINT_KEY>'
if not api_key:
    raise Exception("A key should be provided to invoke the endpoint")

headers = {'Content-Type':'application/json', 'Accept': 'application/json', 'Authorization':('Bearer '+ api_key)}

req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)

    result = response.read()
    print(result)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))

    # Print the headers - they include the request ID and the timestamp, which are useful for debugging the failure
    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

b'[0]'


## Test 2 — Known Fraudulent Transaction

Expected model output: `1` (fraud).

In [ ]:
data = {
  "input_data": {
    "columns": [
      "Time", "V1", "V2", "V3", "V4", "V5", "V6", "V7", "V8", "V9", "V10",
      "V11", "V12", "V13", "V14", "V15", "V16", "V17", "V18", "V19", "V20",
      "V21", "V22", "V23", "V24", "V25", "V26", "V27", "V28", "Amount"
    ],
    "index": [0],
    "data": [[
      406, -2.312226542, 1.951992011, -1.609850732, 3.997905588, -0.522187865,
      -1.426545319, -2.537387306, 1.391657248, -2.770089277, -2.772272145,
      3.202033207, -2.899907388, -0.595221881, -4.289253782, 0.38972412,
      -1.14074718, -2.830055675, -0.016822468, 0.416955705, 0.126910559,
      0.517232371, -0.035049369, -0.465211076, 0.320198199, 0.044519167,
      0.177839798, 0.261145003, -0.143275875, 0
    ]]
  },
  "params": {}
}

body = str.encode(json.dumps(data))

req = urllib.request.Request(url, body, headers)

try:
    response = urllib.request.urlopen(req)

    result = response.read()
    print(result)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))
    print(error.info())
    print(error.read().decode("utf8", 'ignore'))

b'[1]'


## Result Summary

| Test | Input | Expected | Actual | Status |
|---|---|---|---|---|
| Legitimate transaction | Sample #1 | `0` | `0` | ✅ Pass |
| Fraudulent transaction | Sample #2 | `1` | `1` | ✅ Pass |

Both scenarios match expected classification. See `docs/05_UAT_Checklist.md` for full sign-off.